# DataCo Supply Chain Analysis

I'm starting by loading the raw supply chain dataset. The file uses `latin1` encoding and contains order item, shipping, customer, and product information.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

raw = pd.read_csv("../raw_unedited/DataCoSupplyChainDataset.csv", encoding="latin1", low_memory=False)
raw.head()

Next I'm checking for duplicate rows and verifying candidate primary keys. The `Order Item Id` uniquely identifies every row in the dataset.

In [ ]:
d = raw.copy()
for c in d.select_dtypes(include=["object", "string"]).columns:
    d[c] = d[c].astype("string").str.strip()
    d[c] = d[c].replace({"": pd.NA})

key = "Order Item Id"
assert d[key].is_unique, "Order Item Id must be unique"
print(f"Verified {len(d):,} unique order items.")

To adhere to data privacy and security best practices, I'm excluding sensitive customer PII fields (`Customer Email`, `Customer Password`, `Customer Fname`, `Customer Lname`, `Customer Street`, `Customer Zipcode`) from the curated export. These are preserved in the raw source but excluded from the downstream analytics model.

In [ ]:
pii_cols = ["Customer Email", "Customer Password", "Customer Fname", "Customer Lname", "Customer Street", "Customer Zipcode"]
d = d.drop(columns=[c for c in pii_cols if c in d.columns])
print(f"Curated schema column count: {len(d.columns)}")

Now I'm parsing the order and shipping timestamps to date formats to enable temporal analysis and calculating sales performance by region.

In [ ]:
d["order date (DateOrders)"] = pd.to_datetime(d["order date (DateOrders)"], format="mixed", errors="coerce")
d["shipping date (DateOrders)"] = pd.to_datetime(d["shipping date (DateOrders)"], format="mixed", errors="coerce")

region_summary = d.groupby("Order Region").agg(
    total_sales=("Sales", "sum"),
    total_profit=("Order Profit Per Order", "sum"),
    late_delivery_rate=("Late_delivery_risk", "mean")
).reset_index().sort_values("total_sales", ascending=False)

print(region_summary.head(10))

plt.figure(figsize=(10, 6))
sns.set_theme(style="whitegrid")
ax = sns.barplot(
    data=region_summary.head(10),
    y="Order Region",
    x="total_sales",
    color="#327a8b"
)
plt.title("Top 10 Order Regions by Total Sales", fontsize=13, weight="bold")
plt.xlabel("Total Sales ($)")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x/1e6:.1f}M"))
plt.tight_layout()
plt.show()